<a href="https://colab.research.google.com/github/sreent/data-management-intro/blob/main/SQL%20Querying/04-lab-sql-querying.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Topic 4 Lab — SQL Querying: For-Each Semantics, Joins, and Composability

This lab practises the predict-first query method on the **Orders & Payments** schema, extended with a sales organisation (offices and employees tables). The purpose is to apply the For-Each model, FWGHOS pipeline, and join-prediction skills on this enriched schema, so that the techniques from the narrative are grounded in hands-on exploration of real join behaviour.

Use the formal vocabulary (row set, For-Each, nested loop, join direction, row multiplication, NULL padding, FWGHOS, CTE) throughout your work.

**Time budget:** Part 1 ~20 min | Part 2 ~15 min | Part 3 ~15 min | Part 4 ~25 min | Part 5 ~20 min | Part 6 ~25 min | **Total ~120 min**

**Tools required:** Google Colab with MySQL (`%%sql` magic)

---

## Setup

Run the following cells in a new Google Colab notebook. The full Orders & Payments schema is created from scratch — 14 tables covering the core transactional domain plus the sales-organisation extension. The focus of this lab is purely on querying.

**Cell 1 — Install and connect:**

In [ ]:
# ── Install and start MySQL server on Colab ─────────────────
!apt-get update -qq && apt-get install -y -qq mysql-server > /dev/null 2>&1
!service mysql start
!mysql -e "ALTER USER 'root'@'localhost' IDENTIFIED WITH mysql_native_password BY '';" 2>/dev/null || true
!mysql -e "SELECT 'MySQL is ready!' AS status;"


In [ ]:
# Topic 4 — SQL Querying: Orders & Payments Lab
!pip install -q ipython-sql==0.5.0
!pip install -q pymysql==1.1.0
!pip install -q sqlalchemy==2.0.20

%load_ext sql


**Cell 2 — Create database, tables, and load seed data:**


In [ ]:
# ── Download DDL and seed data from course repository ─────
import urllib.request, os

BASE_URL = "https://raw.githubusercontent.com/sreent/data-management-intro/main/resources/orders-payments/"

for fname in ["ddl.sql", "seed-data.sql"]:
    if not os.path.exists(fname):
        urllib.request.urlretrieve(BASE_URL + fname, fname)
        print(f"Downloaded {fname} ({os.path.getsize(fname):,} bytes)")

# Drop existing database (safe re-run)
!mysql -e "DROP DATABASE IF EXISTS orders_payments;"
# Run DDL (creates database, tables, and triggers)
!mysql < ddl.sql
# Load seed data
!mysql < seed-data.sql
print("Database ready.")

# Connect to the database
%sql mysql+pymysql://root:@localhost/orders_payments


**Cell 4 — Verify setup:**

In [ ]:
%%sql
SELECT 'Employees' AS entity, COUNT(*) AS cnt FROM employees
UNION ALL SELECT 'Offices', COUNT(*) FROM offices
UNION ALL SELECT 'Customers', COUNT(*) FROM customers
UNION ALL SELECT 'Orders', COUNT(*) FROM orders
UNION ALL SELECT 'Order Lines', COUNT(*) FROM order_lines
UNION ALL SELECT 'Products', COUNT(*) FROM products;
-- Expected: 8 employees, 5 offices, 5 customers, 7 orders, 13 order lines, 6 products


**Schema Diagram:**

```
offices ──< employees ──< customers ──< orders ──< order_lines >── products
              │  ↑                      │
              │  └── (manager_id)   payments
              │                     shipments
              │                     reviews
              └── customers.sales_rep_id

products >──< categories        (via product_categories junction)
products >──< suppliers        (via supplier_products junction)
```

Key relationships:
- `employees.manager_id -> employees.employee_id` (self-reference; Sarah Tan has NULL — no manager)
- `employees.office_id -> offices.office_id` (every employee belongs to one office)
- `customers.sales_rep_id -> employees.employee_id` (nullable — Eve has no sales rep)
- `customers.postcode -> postcodes.postcode` (nullable — Eve has no postcode)
- `orders.customer_id -> customers.customer_id`
- `order_lines.(order_id, product_id)` — composite PK; FK to both orders and products
- `payments.order_id -> orders.order_id` (one payment per order; Order 106 has none)
- Plus the M:N junctions: product_categories, supplier_products

---

## Part 1 — For-Each and Single-Table Queries (Warm-Up)

### Q1. For-Each with filter

List all employees in office 1 (Singapore).

**(a)** Write the For-Each pseudocode first:

```
FOR EACH row e IN employees:
    IF e.office_id = 1:
        OUTPUT (e.employee_id, e.first_name, e.last_name, e.job_title)
```

**(b)** Write and run the SQL equivalent.

**(c)** How many rows did you get? Does this match the pseudocode's prediction?

---

### Q2. DISTINCT

**(a)** *Predict before running:* The employees table has 8 rows. How many rows should `SELECT DISTINCT job_title FROM employees` return? (Hint: some employees share the same title.)

**(b)** Run the query. Does it match your prediction?

**(c)** This query returns job titles but not the office where each employee works. Why would you need a join to get that information? (We return to this in Q6.)

---

### Q3. ORDER BY and NULL

**(a)** List all employees sorted by `commission_pct` ascending. Run:

In [ ]:
%%sql
SELECT first_name, last_name, commission_pct
FROM employees
ORDER BY commission_pct;


**(b)** Observe: where do employees with NULL `commission_pct` appear? (MySQL places NULLs first in ASC order.)

**(c)** How would you push NULLs to the bottom? Try:

In [ ]:
%%sql
SELECT first_name, last_name, commission_pct
FROM employees
ORDER BY commission_pct IS NULL, commission_pct;


**(d)** Explain why this works. (Hint: `commission_pct IS NULL` evaluates to 0 for non-NULL rows and 1 for NULL rows. Sorting by this expression first pushes NULLs to the end.)

---

### Q4. NULL in WHERE

**(a)** *Predict before running:* How many rows does this return?

In [ ]:
%%sql
SELECT first_name, last_name FROM employees WHERE commission_pct != NULL;


**(b)** Run it. Explain why it returns zero rows. (Hint: any comparison with NULL evaluates to UNKNOWN.)

**(c)** Write the correct version using `IS NULL` to find employees without a commission.

---

### Q5. COUNT and NULL

**(a)** Run:

In [ ]:
%%sql
SELECT COUNT(*) AS total_employees,
       COUNT(commission_pct) AS with_commission
FROM employees;


**(b)** What does the difference `COUNT(*) - COUNT(commission_pct)` represent? How does this relate to `WHERE commission_pct IS NULL`?

---

## Part 2 — Multi-Table Inner Joins (Predict-First)

### Q6. Two-table join (many-to-one)

For each employee, show their office city.

**(a)** *Predict before running:* employees has 8 rows. offices has 5 rows. The join is on `office_id` (FK in employees -> PK in offices). Direction: many-to-one — each employee matches exactly one office. Predicted output: ___ rows.

**(b)** Write and run the query.

**(c)** Does the row count match your prediction?

---

### Q7. Three-table join (chain)

For each customer, show their sales rep's name and the office city where the rep is based.

**(a)** *Predict before running:* The join chain is customers -> employees -> offices. Each join is many-to-one. But `customers.sales_rep_id` is nullable — what happens to customers with no sales rep?

**(b)** Write and run:

In [ ]:
%%sql
SELECT c.name AS customer, e.first_name, e.last_name AS rep,
       o.city AS office
FROM customers c
JOIN employees e ON c.sales_rep_id = e.employee_id
JOIN offices o ON e.office_id = o.office_id;


**(c)** Count the result. Is it fewer than 5? If so, which customer was dropped and why? Check with: `SELECT * FROM customers WHERE sales_rep_id IS NULL;`

---

## Part 3 — Self-Join (For-Each Limitation)

### Q8. Self-join — employee with manager

**(a)** Write the For-Each pseudocode for: "list each employee alongside their manager's name":

```
FOR EACH row e IN employees:
    FOR EACH row m IN employees:
        IF e.manager_id = m.employee_id:
            OUTPUT (e.first_name, e.last_name, m.first_name, m.last_name)
```

**(b)** Write and run the SQL using LEFT JOIN:

In [ ]:
%%sql
SELECT e.first_name, e.last_name,
       m.first_name AS manager_first, m.last_name AS manager_last
FROM employees e
LEFT JOIN employees m ON e.manager_id = m.employee_id;


**(c)** *Predict before checking:* How many rows? Which employee has NULL for manager columns? Why LEFT JOIN instead of INNER JOIN?

---

### Q9. Self-join — salary comparison

List employees who earn more than their manager.

**(a)** Write the SQL. (Hint: INNER JOIN this time — if the manager does not exist, the salary comparison is meaningless.)

**(b)** *Predict:* Will there be many such employees, or few? Why?

**(c)** Run the query and review the results. Are they surprising?

---

## Part 4 — LEFT JOIN, Anti-Join, and Row Multiplication

### Q10. LEFT JOIN — preserving empty offices

List all offices and the number of employees in each.

**(a)** *Predict before running:* offices has 5 rows. Some offices may have zero employees. What does INNER JOIN do to empty offices?

**(b)** First attempt — use COUNT(*):

In [ ]:
%%sql
SELECT o.city, COUNT(*) AS headcount
FROM offices o
LEFT JOIN employees e ON o.office_id = e.office_id
GROUP BY o.office_id, o.city
ORDER BY headcount DESC;


**(c)** Check: do any offices show headcount 0? If not, why does the headcount show 1 for the empty office?

**(d)** Fix: replace `COUNT(*)` with `COUNT(e.employee_id)`. Run again. Does the empty office now show 0?

**(e)** Explain the difference between `COUNT(*)` and `COUNT(e.employee_id)` after a LEFT JOIN.

---

### Q11. Row multiplication

For each employee, show how many customers they serve and how many orders those customers have placed.

**(a)** *Predict before running:* What happens when a customer has 2 orders? The LEFT JOIN from customers to orders multiplies that customer's row. How does this affect COUNT(*)?

**(b)** Run:

In [ ]:
%%sql
SELECT e.first_name, e.last_name,
       COUNT(*) AS total_rows,
       COUNT(DISTINCT c.customer_id) AS customer_count,
       COUNT(o.order_id) AS order_count
FROM employees e
LEFT JOIN customers c ON e.employee_id = c.sales_rep_id
LEFT JOIN orders o ON c.customer_id = o.customer_id
GROUP BY e.employee_id, e.first_name, e.last_name;


**(c)** Compare the three COUNT columns. For which employees is `COUNT(*)` misleading? Why?

**(d)** Why is `COUNT(DISTINCT c.customer_id)` correct for customer count but `COUNT(o.order_id)` (without DISTINCT) correct for order count?

---

### Q12. Left anti-join — employees with no customers

**(a)** Try `NOT IN`:

In [ ]:
%%sql
SELECT e.employee_id, e.first_name, e.last_name
FROM employees e
WHERE e.employee_id NOT IN (SELECT sales_rep_id FROM customers);


*Predict:* Does this return the employees who serve no customers? Or could it return zero rows? Check: `SELECT COUNT(*) FROM customers WHERE sales_rep_id IS NULL;`

**(b)** Write the left anti-join version:

In [ ]:
%%sql
SELECT e.employee_id, e.first_name, e.last_name
FROM employees e
LEFT JOIN customers c ON e.employee_id = c.sales_rep_id
WHERE c.customer_id IS NULL;


**(c)** Compare results. If `NOT IN` returned zero rows while the anti-join returned employees, explain why (the NULL trap).

**(d)** Why filter on `c.customer_id IS NULL` rather than `c.sales_rep_id IS NULL`?

---

## Part 5 — Grouping, Aggregation, HAVING, and DISTINCT

### Q13. GROUP BY + HAVING

Find offices where the average employee salary exceeds 40,000.

**(a)** *Predict:* How many of the 5 offices will survive the HAVING filter? (Rough estimate is fine.)

**(b)** Run:

In [ ]:
%%sql
SELECT o.city, ROUND(AVG(e.salary), 2) AS avg_salary
FROM offices o
JOIN employees e ON o.office_id = e.office_id
GROUP BY o.office_id, o.city
HAVING AVG(e.salary) > 40000
ORDER BY avg_salary DESC;


**(c)** How many survived? Was your prediction close?

---

### Q14. WHERE vs HAVING error

**(a)** A colleague writes:

In [ ]:
%%sql
SELECT office_id, AVG(salary)
FROM employees
WHERE AVG(salary) > 40000
GROUP BY office_id;


*Predict:* What error does this produce?

**(b)** Trace the FWGHOS pipeline to explain *why* this fails. At which step does the error occur?

**(c)** Fix it by moving the condition to HAVING.

---

### Q15. COUNT vs COUNT(DISTINCT)

How many distinct job titles exist in each office?

**(a)** Run both queries:

In [ ]:
%%sql
-- Query A
SELECT o.city, COUNT(e.job_title) AS title_count
FROM offices o
JOIN employees e ON o.office_id = e.office_id
GROUP BY o.office_id, o.city;

-- Query B
SELECT o.city, COUNT(DISTINCT e.job_title) AS distinct_titles
FROM offices o
JOIN employees e ON o.office_id = e.office_id
GROUP BY o.office_id, o.city;


**(b)** Why do the numbers differ? What does each count?

**(c)** Which is the correct answer to "how many different roles exist in each office?"

---

## Part 6 — CTEs and the Chapter 3 Loop-Back

### Q16. CTE — above-average salary per office

Find employees whose salary is above the average for their office.

**(a)** Write the CTE:

In [ ]:
%%sql
WITH office_avg AS (
    SELECT office_id, AVG(salary) AS avg_salary
    FROM employees
    GROUP BY office_id
)
SELECT e.first_name, e.last_name, e.salary,
       o.city, ROUND(oa.avg_salary, 2) AS office_avg
FROM employees e
JOIN offices o ON e.office_id = o.office_id
JOIN office_avg oa ON e.office_id = oa.office_id
WHERE e.salary > oa.avg_salary
ORDER BY o.city, e.salary DESC;


**(b)** Could you write this without a CTE? (Yes — using a correlated subquery. But the CTE version is easier to read and extend.)

---

### Q17. CTE — multi-step aggregation across the full chain

Find each sales territory's total order revenue and the percentage of overall tracked revenue it represents.

**(a)** *Predict before running:* The join chain is employees -> offices (for territory) and employees -> customers -> orders -> order_lines (for revenue). Each INNER JOIN drops employees without customers — how many employees contribute revenue?

**(b)** Write and run:

In [ ]:
%%sql
WITH territory_revenue AS (
    SELECT o.territory, SUM(ol.quantity * ol.unit_price) AS total_revenue
    FROM employees e
    JOIN offices o ON e.office_id = o.office_id
    JOIN customers c ON e.employee_id = c.sales_rep_id
    JOIN orders ord ON c.customer_id = ord.customer_id
    JOIN order_lines ol ON ord.order_id = ol.order_id
    GROUP BY o.territory
)
SELECT territory, total_revenue,
       ROUND(total_revenue * 100.0 / (SELECT SUM(total_revenue) FROM territory_revenue), 1) AS pct
FROM territory_revenue
ORDER BY total_revenue DESC;


**(c)** The CTE `territory_revenue` is referenced twice — once in the main SELECT (to list territories) and once in the subquery (to compute the denominator). What would the query look like without a CTE? How many times would the 5-table join be duplicated?

---

### Q18. Chapter 3 loop-back — written analysis

In Chapter 3, you ran a multi-table reporting query against the BCNF schema. Now explain a similar query using this chapter's vocabulary:

In [ ]:
%%sql
SELECT cat.name AS category,
       QUARTER(o.order_date) AS q,
       YEAR(o.order_date) AS y,
       SUM(ol.quantity * ol.unit_price) AS revenue
FROM order_lines ol
  JOIN products p    ON ol.product_id = p.product_id
  JOIN product_categories pc ON p.product_id = pc.product_id
  JOIN categories cat ON pc.category_id = cat.category_id
  JOIN orders o     ON ol.order_id = o.order_id
GROUP BY cat.name, YEAR(o.order_date), QUARTER(o.order_date)
ORDER BY y, q, revenue DESC;


Answer these questions in writing (no query to run):

**(a)** List every join in the FROM clause and the direction of each (many-to-one or one-to-many). Which joins preserve the row count? Could any expand it?

**(b)** Trace the FWGHOS pipeline: what does each step produce?

**(c)** Why does SUM work correctly here — even though order_lines was joined through multiple tables?

**(d)** Compare to the star schema version from Chapter 3 (`order_line_facts JOIN product_dim JOIN time_dim`). Why does the star schema query have fewer joins but produce the same result?

---

## Deliverable

- **Part 1** (Q1--Q5): For-Each pseudocode for Q1, NULL observations from Q3--Q5.
- **Part 2** (Q6--Q7): Row-count predictions and verification for both joins.
- **Part 3** (Q8--Q9): For-Each pseudocode for Q8, LEFT vs INNER JOIN reasoning.
- **Part 4** (Q10--Q12): COUNT(*) vs COUNT(column) observation, anti-join comparison from Q12.
- **Part 5** (Q13--Q15): WHERE vs HAVING error explanation, COUNT vs COUNT(DISTINCT) comparison.
- **Part 6** (Q16--Q18): CTE queries and written analysis of the Chapter 3 reporting query.

---

## Solutions

<details>
<summary><strong>Q1 Solution</strong></summary>

**(b)**
```sql
%%sql
SELECT employee_id, first_name, last_name, job_title
FROM employees
WHERE office_id = 1;
```



**(c)** 4 rows (Sarah Tan, Wei Jie Lim, Mei Ling Wong, Nurul Huda). The pseudocode predicts: visit all 8 rows, keep those where `office_id = 1`. Four match.
</details>

---

<details>
<summary><strong>Q2 Solution</strong></summary>

**(a)** Fewer than 8. Several employees share the title "Sales Rep" (Ahmad, Mei Ling, Somchai, Nurul). The distinct titles are: Managing Director, Sales Manager SG, Sales Manager ID, Sales Rep, Sales Rep ID — that is 5 distinct titles.

**(b)** The query returns 5 rows.

**(c)** `SELECT DISTINCT job_title` returns titles but no office information. The office city is stored in the `offices` table. Getting both the title and the city requires joining employees to offices on `office_id`.
</details>

---

<details>
<summary><strong>Q3 Solution</strong></summary>

**(b)** NULL values appear first in ascending order (MySQL default). Sarah, Wei Jie, and Budi (all with NULL `commission_pct`) appear at the top.

**(d)** `commission_pct IS NULL` evaluates to 1 (true) for NULL rows and 0 (false) for non-NULL rows. Sorting by this expression first groups non-NULL rows (0) before NULL rows (1). Within each group, `commission_pct` sorts normally ascending. The result: Rizal (0.10), Mei Ling (0.12), Ahmad (0.15), Nurul (0.15), Somchai (0.18), then the three NULLs.
</details>

---

<details>
<summary><strong>Q4 Solution</strong></summary>

**(a)** Zero rows.

**(b)** `!= NULL` evaluates to UNKNOWN for every row — even rows where `commission_pct` IS NULL. WHERE treats UNKNOWN as false, so no rows pass. The correct comparison is `IS NULL` / `IS NOT NULL`.

**(c)** `SELECT first_name, last_name FROM employees WHERE commission_pct IS NULL;` — returns 3 rows (Sarah, Wei Jie, Budi).
</details>

---

<details>
<summary><strong>Q5 Solution</strong></summary>

**(b)** `COUNT(*)` returns 8 (all employees). `COUNT(commission_pct)` returns 5 (employees with a non-NULL commission). The difference (3) is the number of employees without a commission — the three managers (Sarah, Wei Jie, Budi). This is equivalent to `SELECT COUNT(*) FROM employees WHERE commission_pct IS NULL`.
</details>

---

<details>
<summary><strong>Q6 Solution</strong></summary>

**(a)** 8 rows — each employee matches exactly one office (many-to-one join on FK -> PK).

**(b)**

In [ ]:
%%sql
SELECT e.first_name, e.last_name, o.city
FROM employees e
JOIN offices o ON e.office_id = o.office_id;


**(c)** Yes, 8 rows. The many-to-one join preserves the row count because every employee has a non-NULL `office_id` that matches exactly one office.
</details>

---

<details>
<summary><strong>Q7 Solution</strong></summary>

**(a)** If every customer has a sales rep, 5 rows. But `customers.sales_rep_id` is nullable — Eve has no sales rep.

**(c)** 4 rows — one fewer than expected. Eve (customer_id = 5) was dropped by the INNER JOIN because her `sales_rep_id` is NULL, which matches no employee. The surviving rows are: Alice (Mei Ling, Singapore), Bob (Ahmad, Kuala Lumpur), Carol (Somchai, Bangkok), Dave (Rizal, Jakarta). To keep Eve, use LEFT JOIN.
</details>

---

<details>
<summary><strong>Q8 Solution</strong></summary>

**(c)** 8 rows — one per employee. Sarah Tan (employee_id = 1) has `manager_id = NULL`, so `m.first_name` and `m.last_name` are NULL for her row. LEFT JOIN preserves her; INNER JOIN would drop her (returning only 7 rows). We use LEFT JOIN because we want to see all employees, including the top-level executive who has no manager.
</details>

---

<details>
<summary><strong>Q9 Solution</strong></summary>

**(a)**

In [ ]:
%%sql
SELECT e.first_name, e.last_name, e.salary AS emp_salary,
       m.first_name AS mgr_first, m.last_name AS mgr_last, m.salary AS mgr_salary
FROM employees e
JOIN employees m ON e.manager_id = m.employee_id
WHERE e.salary > m.salary;


INNER JOIN because we need both salaries to compare — if no manager exists, the comparison is meaningless.

**(b)** Few. Most employees earn less than their manager.

**(c)** 1 row: Rizal Pratama (56,000) earns more than his manager Budi Santoso (55,000). This is the only case in the data where a report's salary exceeds the manager's.
</details>

---

<details>
<summary><strong>Q10 Solution</strong></summary>

**(c)** No office shows headcount 0. Ho Chi Minh City (which has no employees) shows 1 because LEFT JOIN produces one NULL-padded row, and `COUNT(*)` counts it.

**(d)** Yes. `COUNT(e.employee_id)` skips NULLs, so the NULL-padded row from Ho Chi Minh City contributes 0 instead of 1. Results: Singapore = 4, Jakarta = 2, Kuala Lumpur = 1, Bangkok = 1, Ho Chi Minh City = 0.

**(e)** `COUNT(*)` counts all rows regardless of content — including NULL-padded rows from LEFT JOIN. `COUNT(column)` counts non-NULL values in that column. After a LEFT JOIN, `COUNT(column)` on a right-table column gives the correct count of actual matches.
</details>

---

<details>
<summary><strong>Q11 Solution</strong></summary>

**(c)** `COUNT(*)` is misleading in two ways:
- For managers with no customers (Sarah, Wei Jie, Budi, Nurul): `COUNT(*) = 1` because of the NULL-padded row from the LEFT JOIN. But they have 0 customers and 0 orders. The "1" is an artefact of NULL padding, not a count of anything meaningful.
- For sales reps whose customers have multiple orders (Ahmad, Mei Ling, Somchai): `COUNT(*) = 2` because the customers-to-orders LEFT JOIN produced two rows (one per order). This matches the order count, but if someone reads it as "customer count" they get the wrong answer — each of these reps serves only 1 customer.

`COUNT(DISTINCT c.customer_id)` correctly returns 0 for managers and 1 for each active sales rep. `COUNT(o.order_id)` correctly returns 0 for managers and the actual order count (1 or 2) for reps.

**(d)** For customer count, each customer should be counted once regardless of how many orders they have — hence DISTINCT. For order count, each `order_id` value in the joined result represents one order, so counting non-NULL values (without DISTINCT) is correct. Using `COUNT(DISTINCT o.order_id)` would also work for orders here, but is unnecessary — each order appears exactly once per employee because each customer has one sales rep. The key is that `COUNT(*)` conflates the NULL-padded rows (from employees without customers) with the order-multiplied rows (from employees with customers), making it unreliable for either count.
</details>

---

<details>
<summary><strong>Q12 Solution</strong></summary>

**(a)** The subquery `SELECT sales_rep_id FROM customers` returns: 5, 4, 6, 7, NULL (Eve's NULL). Because of the NULL, `NOT IN` returns **zero rows** — the NULL trap. `employee_id NOT IN (5, 4, 6, 7, NULL)` evaluates to UNKNOWN for every employee, and WHERE discards UNKNOWN.

**(b)** The left anti-join version correctly returns 4 employees: Sarah (1), Wei Jie (2), Budi (3), and Nurul (8) — the employees who serve no customers.

**(c)** `NOT IN` returns 0 rows; the anti-join returns 4 rows. The single NULL in the subquery (from Eve's `sales_rep_id`) breaks `NOT IN`. Why? Because `x NOT IN (5, 4, 6, 7, NULL)` evaluates as `x != 5 AND x != 4 AND x != 6 AND x != 7 AND x != NULL`. The last term is always UNKNOWN, making the entire expression UNKNOWN for every x.

**(d)** `c.sales_rep_id` could be NULL for a *matched* row (Eve's record, if the LEFT JOIN happened to match on some other condition). `c.customer_id` (the PK of customers) is only NULL when the LEFT JOIN found no match at all — i.e., when the employee serves no customer. Filtering on the PK correctly identifies truly unmatched rows.
</details>

---

<details>
<summary><strong>Q13 Solution</strong></summary>

**(a)** Rough estimate: 2 or 3 offices. Only offices with higher-paid roles (directors, managers) would have averages above 40,000.

**(c)** 2 offices survive:
- Singapore: AVG(85000, 52000, 38000, 33000) = 52,000
- Jakarta: AVG(55000, 56000) = 55,500

The remaining offices fall below 40,000: Kuala Lumpur (34,000), Bangkok (31,000). Ho Chi Minh City has no employees, so it is excluded by the INNER JOIN (no group is formed).
</details>

---

<details>
<summary><strong>Q14 Solution</strong></summary>

**(a)** MySQL error: "Invalid use of group function."

**(b)** FWGHOS trace: at step W (WHERE), the engine is filtering individual rows. GROUP BY has not run yet — groups do not exist. There is nothing to compute `AVG()` over. The aggregate requires groups, which are created at step G.

**(c)** Move to HAVING:

In [ ]:
%%sql
SELECT office_id, AVG(salary)
FROM employees
GROUP BY office_id
HAVING AVG(salary) > 40000;

</details>

---

<details>
<summary><strong>Q15 Solution</strong></summary>

**(b)** Query A (`COUNT(e.job_title)`) counts the total number of employees per office — since every employee has a `job_title`, this is equivalent to headcount. Query B (`COUNT(DISTINCT e.job_title)`) counts unique job titles per office — this answers "how many different roles exist."

For Singapore: Query A returns 4 (four employees), Query B returns 3 (Managing Director, Sales Manager SG, Sales Rep — Mei Ling and Nurul both hold "Sales Rep," which DISTINCT collapses to one).

**(c)** Query B. An office with 2 Sales Reps and 1 Sales Manager has `COUNT(job_title) = 3` but `COUNT(DISTINCT job_title) = 2`.
</details>

---

<details>
<summary><strong>Q16 Solution</strong></summary>

**(a)** The query returns 2 employees:
- Sarah Tan (Singapore): salary 85,000 exceeds the Singapore average of 52,000.
- Rizal Pratama (Jakarta): salary 56,000 exceeds the Jakarta average of 55,500.

All other employees either match or fall below their office average. Kuala Lumpur and Bangkok each have one employee, so no employee can be above the single-person average.

**(b)** Yes, using a correlated subquery: `WHERE e.salary > (SELECT AVG(e2.salary) FROM employees e2 WHERE e2.office_id = e.office_id)`. The CTE version is clearer because the average computation is named and visible at the top of the query, and it could be referenced multiple times without duplication.
</details>

---

<details>
<summary><strong>Q17 Solution</strong></summary>

**(a)** Only 4 employees contribute revenue — the sales reps who have customers: Mei Ling (Alice), Ahmad (Bob), Somchai (Carol), Rizal (Dave). The three managers and Nurul (who has no customers) are excluded by the INNER JOIN on `c.sales_rep_id`. Eve (no sales rep) is also excluded. Since Eve has 0 orders, no revenue is lost.

The 5-table join chain: employees -> offices -> customers -> orders -> order_lines. Each join is many-to-one from order_lines's perspective, so the joined row set has 13 rows (one per order line from the 4 active reps' customers).

**(b)** Result: 4 territories.
- Malaysia: 159.95 (40.5%) — Kuala Lumpur (Ahmad -> Bob: 159.95)
- Thailand: 119.94 (30.4%) — Bangkok (Somchai -> Carol: 119.94)
- Singapore: 84.96 (21.5%) — Singapore (Mei Ling -> Alice: 84.96)
- Indonesia: 29.96 (7.6%) — Jakarta (Rizal -> Dave: 29.96)

Total tracked revenue: $394.81. Percentages sum to 100.0%.

**(c)** Without a CTE, the 5-table join and GROUP BY would appear twice: once in the main SELECT (to list territories) and once in the subquery (to compute the denominator sum). That is 10 JOINs total instead of 5. If the business logic changes — for example, excluding pending orders — both copies must be updated. Forgetting one is exactly Failure 4 from the narrative.
</details>

---

<details>
<summary><strong>Q18 Solution</strong></summary>

**(a)** Join directions (all from order_lines's perspective):
- `order_lines JOIN products ON product_id` — many-to-one (FK -> PK). Each line item matches one product. Row count preserved (13 rows).
- `products JOIN product_categories ON product_id` — one-to-many (PK <- FK in junction table). Each product could match multiple categories. **This join could multiply rows.** In our data, each product has exactly one category, so no multiplication occurs (still 13 rows). But with products in multiple categories, SUM would double-count.
- `product_categories JOIN categories ON category_id` — many-to-one (FK -> PK). Row count preserved.
- `order_lines JOIN orders ON order_id` — many-to-one (FK -> PK). Each line item matches one order. Row count preserved.

**(b)** FWGHOS trace:
- **F:** 13 order_lines rows joined through products, product_categories, categories, and orders. All joins preserve the row count (in this data) -> 13 joined rows, each carrying category name and order date.
- **W:** No WHERE clause — all 13 rows survive.
- **G:** GROUP BY category name, year, quarter. Creates groups: Peripherals Q1, Peripherals Q2, Peripherals Q3, Peripherals Q4, Cables Q2, Cables Q3, Cables Q4 -> 7 groups.
- **H:** No HAVING — all 7 groups survive.
- **O:** ORDER BY year, quarter, revenue DESC.
- **S:** Output category, quarter, year, SUM(quantity x unit_price).

**(c)** SUM works correctly because the joins did not multiply rows — each order_lines row appears exactly once in the joined result (in this data). The many-to-one joins attach attributes (product name, category, order date) without changing the row count. SUM aggregates these unique rows within each group. However, the product_categories junction join is a risk: if any product belonged to multiple categories, the join would multiply that product's order lines, and SUM would overcount revenue. Awareness of this risk is the prediction skill from Section 3.3 of the narrative.

**(d)** The star schema's `product_dim` already contains `category_name` — the joins through `products -> product_categories -> categories` are pre-flattened into a single dimension table. The M:N junction is resolved at ETL time, not at query time. The FWGHOS pipeline is identical in structure (F -> W -> G -> H -> O -> S), but step F has 2 joins instead of 4 — and no risk of row multiplication from the M:N junction.
</details>